In [1]:
# -*- coding: utf-8 -*-
"""
完整三模态 + 超图嵌入融合 Prompt 构建
保留原始所有信息：Edge List + GCN Embedding + Opcode Stats + Sequence + HGCNN Embedding
"""

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import train_test_split
import json
from collections import Counter, defaultdict
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
# ========================================
# 1. 数据读取与清理
# ========================================
FILE_PATH = 'E:\\ponzi\\data\\2PonziContract_20220701.csv'
ADDRESS_COLUMN = 'address'
LABEL_COLUMN = 'label'

def clean_and_analyze_data(df: pd.DataFrame, addr_col: str, label_col: str) -> pd.DataFrame:
    original_len = len(df)
    label_counts = df.groupby(addr_col)[label_col].nunique()
    inconsistent_addresses = label_counts[label_counts > 1].index.tolist()
    df_step1 = df[~df[addr_col].isin(inconsistent_addresses)].copy()
    df_cleaned = df_step1.drop_duplicates(subset=[addr_col], keep='first').copy()
    print(f"清理后样本数: {len(df_cleaned)}")
    return df_cleaned

df = pd.read_csv(FILE_PATH)
ponzi_contract = clean_and_analyze_data(df, ADDRESS_COLUMN, LABEL_COLUMN)

# ========================================
# 2. 操作码 Chunking & 符号化
# ========================================
START_OPS = {f'PUSH{i}' for i in range(1, 33)}
END_OPS = {'MSTORE', 'SSTORE', 'CALLDATACOPY', 'RETURN', 'STOP', 'JUMP', 'JUMPI', 'JUMPDEST', 'SELFDESTRUCT'}
BLOCK_TYPE_MAPPING = {
    "Data Preparation/Stack Building Block": "<DATA>",
    "Storage/Memory Write Block": "<WRITE>",
    "Control Flow Block": "<CTRL>",
    "Hashing Block": "<HASH>",
    "Complex Call/Interaction Block": "<CALL>",
    "Computation/Logic Block": "<COMP>",
    "Calldata Copy Block": "<COPY>",
    "Jump Destination Block": "<DEST>",
}

def chunk_opcodes_list_classified(opcode_list: list[str]) -> list[dict]:
    chunks_data, current_chunk = [], []
    current_chunk_start_index, n, i = None, len(opcode_list), 0
    while i < n:
        op_code = opcode_list[i]
        is_push, is_term = op_code in START_OPS, op_code in END_OPS

        if is_push:
            if current_chunk:
                chunks_data.append({'chunk': current_chunk, 'type': classify_chunk(current_chunk), 'start_index': current_chunk_start_index})
                current_chunk = []
            current_chunk_start_index = i
            current_chunk.append(op_code)
            i += 1
            push_data_len = int(op_code[4:]) if op_code.startswith('PUSH') and op_code[4:].isdigit() else 0
            for _ in range(push_data_len):
                if i < n: current_chunk.append(opcode_list[i]); i += 1
                else: break
            continue

        elif not is_term:
            if current_chunk: current_chunk.append(op_code)
            i += 1
            continue

        elif is_term:
            if not current_chunk: current_chunk_start_index = i
            current_chunk.append(op_code)
            chunks_data.append({'chunk': current_chunk, 'type': classify_chunk(current_chunk), 'start_index': current_chunk_start_index})
            current_chunk = []
            current_chunk_start_index = None
            i += 1
            continue

    if current_chunk:
        chunks_data.append({'chunk': current_chunk, 'type': classify_chunk(current_chunk), 'start_index': current_chunk_start_index})
    return chunks_data

def classify_chunk(chunk):
    terminal_op = chunk[-1]
    if terminal_op in {'JUMP', 'JUMPI', 'JUMPDEST', 'RETURN', 'STOP', 'SELFDESTRUCT'}:
        if 'JUMPDEST' in chunk and len(chunk) == 1: return "Jump Destination Block"
        return "Control Flow Block"
    if terminal_op in {'MSTORE', 'SSTORE'}: return "Storage/Memory Write Block"
    if terminal_op == 'CALLDATACOPY': return "Calldata Copy Block"
    for op in chunk:
        if op in {'CALL', 'DELEGATECALL', 'STATICCALL', 'CALLCODE', 'CREATE', 'CREATE2', 'LOG0', 'LOG1', 'LOG2', 'LOG3', 'LOG4'}:
            return "Complex Call/Interaction Block"
        if op == 'SHA3': return "Hashing Block"
    if chunk and chunk[0] in START_OPS: return "Data Preparation/Stack Building Block"
    return "Computation/Logic Block"

def analyze_chunk_statistics(chunks_data):
    type_counter = Counter()
    length_stats = defaultdict(list)
    for c in chunks_data:
        block_type, block_len = c['type'], len(c['chunk'])
        type_counter[block_type] += 1
        length_stats[block_type].append(block_len)
    summary = []
    for block_type, count in type_counter.items():
        lengths = length_stats[block_type]
        avg_len = sum(lengths) / count if count > 0 else 0
        summary.append({
            'type': block_type, 'count': count, 'avg_length': round(avg_len, 2),
            'max_length': max(lengths) if lengths else 0, 'min_length': min(lengths) if lengths else 0
        })
    return summary

def generate_opcode_mapping_sequence(chunks_data):
    return "".join(BLOCK_TYPE_MAPPING.get(c['type'], "<UNK>") for c in chunks_data)

# ========================================
# 3. Trace-Chain 图构建 + 边列表提取
# ========================================
def construct_trace_graph_with_edges(ponzi_contract, addr):
    opcode = ponzi_contract[ponzi_contract['address'] == addr]['opcode'].values[0]
    opcode_lines = opcode.split('\n')
    opcode_sequence_str = opcode_lines[0]
    opcode_list = [op for op in opcode_sequence_str.split(' ') if op]

    # 构建边
    trace_data = []
    edge_list_str = []
    for i in range(len(opcode_list) - 1):
        from_op = opcode_list[i]
        to_op = opcode_list[i + 1]
        position = i + 1
        trace_data.append((from_op, to_op, position))
        edge_list_str.append(f"{i}->{i+1}")

    df = pd.DataFrame(trace_data, columns=['from_opcode_id', 'to_opcode_id', 'position'])

    # one-hot
    all_opcodes = pd.unique(df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    for opcode in all_opcodes:
        col_name = f'opcode_{opcode}'
        df[col_name] = ((df['from_opcode_id'] == opcode) | (df['to_opcode_id'] == opcode)).astype(int)

    # 编码
    unique_addresses = pd.unique(df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    address_encoding = {address: idx for idx, address in enumerate(unique_addresses)}
    df['from'] = df['from_opcode_id'].map(address_encoding)
    df['to'] = df['to_opcode_id'].map(address_encoding)
    df = df.drop(columns=['from_opcode_id', 'to_opcode_id'])

    edge_list_str = " ".join(edge_list_str) if edge_list_str else ""
    return df, edge_list_str

# ========================================
# 4. Hypergraph 构建（HGCNN）
# ========================================
opcode_categories = {
    "Stack Operations": ["POP", "PUSH1", "PUSH2", "PUSH3", "PUSH4", "PUSH5", "PUSH6", "PUSH7", "PUSH8", "PUSH9", "PUSH10", "PUSH11", "PUSH12", "PUSH13", "PUSH14", "PUSH15", "PUSH16", "PUSH17", "PUSH18", "PUSH19", "PUSH20", "PUSH21", "PUSH22", "PUSH23", "PUSH24", "PUSH25", "PUSH26", "PUSH27", "PUSH28", "PUSH29", "PUSH30", "PUSH31", "PUSH32", "DUP1", "DUP2", "DUP3", "DUP4", "DUP5", "DUP6", "DUP7", "DUP8", "DUP9", "DUP10", "DUP11", "DUP12", "DUP13", "DUP14", "DUP15", "DUP16", "SWAP1", "SWAP2", "SWAP3", "SWAP4", "SWAP5", "SWAP6", "SWAP7", "SWAP8", "SWAP9", "SWAP10", "SWAP11", "SWAP12", "SWAP13", "SWAP14", "SWAP15", "SWAP16"],
    "Data Storage and Memory Operations": ["SLOAD", "SSTORE", "MLOAD", "MSTORE", "MSTORE8", "CODECOPY", "EXTCODECOPY"],
    "Mathematics and Logical Operations": ['ADD', 'MUL', 'SUB', 'DIV', 'SDIV', 'MOD', 'SMOD', 'EXP', 'SIGNEXTEND', 'LT', 'GT', 'SLT', 'SGT', 'EQ', 'AND', 'OR', 'XOR', 'NOT', 'BYTE', 'SHL', 'SHR', 'SAR', 'ISZERO'],
    "Control Flow Operations": ["JUMP", "JUMPI", "JUMPDEST", "STOP", "RETURN", "REVERT", "SELFDESTRUCT", "PC"],
    "Contract Call and Message Operations": ["CALL", "CALLCODE", "DELEGATECALL", "STATICCALL", "CREATE", "CREATE2", "SELFDESTRUCT", "RETURN", "REVERT", "STOP"],
    "Environment and State Information": ["GAS", "GASPRICE", "GASLIMIT", "TIMESTAMP", "NUMBER", "DIFFICULTY", "BASEFEE", "COINBASE", "BLOCKHASH", "ADDRESS", "BALANCE", "ORIGIN", "CALLER", "CALLVALUE", "CALLDATALOAD", "CALLDATASIZE", "CALLDATACOPY", "CODESIZE", "CODECOPY", "EXTCODESIZE", "EXTCODECOPY", "EXTCODEHASH", "MSIZE", "RETURNDATASIZE", "RETURNDATACOPY", "PC"],
    "Data Processing and Hashing": ["SHA3", "CALLDATALOAD", "CALLDATASIZE", "CALLDATACOPY", "RETURNDATASIZE", "RETURNDATACOPY", "CODECOPY", "EXTCODECOPY"],
    "Events and Logs Operations": ["LOG0", "LOG1", "LOG2", "LOG3", "LOG4"],
    "Exception and Termination": ["STOP", "RETURN", "REVERT", "INVALID", "SELFDESTRUCT"]
}

def construct_hypergraph(ponzi_contract, addr, opcode_categories):
    opcode = ponzi_contract[ponzi_contract['address'] == addr]['opcode'].values[0]
    opcode_lines = opcode.split('\n')
    opcode_sequence_str = opcode_lines[0]
    opcode_list = [op for op in opcode_sequence_str.split(' ') if op]
    trace_data = []
    for i in range(len(opcode_list) - 1):
        from_op = opcode_list[i]
        to_op = opcode_list[i + 1]
        position = i + 1
        trace_data.append((from_op, to_op, position))
    df = pd.DataFrame(trace_data, columns=['from_opcode_id', 'to_opcode_id', 'position'])

    all_opcodes = pd.unique(df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    for opcode in all_opcodes:
        col_name = f'opcode_{opcode}'
        df[col_name] = ((df['from_opcode_id'] == opcode) | (df['to_opcode_id'] == opcode)).astype(int)

    unique_addresses = pd.unique(df[['from_opcode_id', 'to_opcode_id']].values.ravel('K'))
    address_encoding = {addr: idx for idx, addr in enumerate(unique_addresses)}
    df['from'] = df['from_opcode_id'].map(address_encoding)
    df['to'] = df['to_opcode_id'].map(address_encoding)
    df = df.drop(columns=['from_opcode_id', 'to_opcode_id'])

    # 超边构建
    nodes = list(set(opcode_list))
    categorized = {category: [] for category in opcode_categories.keys()}
    categorized['Other'] = []
    for op in nodes:
        found = False
        for category, op_list in opcode_categories.items():
            if op in op_list:
                encoded_op = address_encoding.get(op, op)
                categorized[category].append(encoded_op)
                found = True
                break
        if not found:
            encoded_op = address_encoding.get(op, op)
            categorized['Other'].append(encoded_op)
    hyperedge_dict = {}
    for category, node_list in categorized.items():
        if node_list:
            hyperedge_dict[category] = [node_list]
    return df, hyperedge_dict

# ========================================
# 5. 节点特征提取
# ========================================
def extract_transactional_features(df):
    all_nodes = pd.Series(df['from'].tolist() + df['to'].tolist()).unique()
    out_degree = df['from'].value_counts().rename('out_degree')
    in_degree = df['to'].value_counts().rename('in_degree')
    features_df = pd.DataFrame(index=all_nodes)
    features_df = features_df.join(out_degree, how='left').fillna(0)
    features_df = features_df.join(in_degree, how='left').fillna(0)
    features_df['total_tx'] = features_df['out_degree'] + features_df['in_degree']
    features_df['net_flow'] = features_df['out_degree'] - features_df['in_degree']

    exclude_cols = ['from', 'to', 'position']
    extra_cols = [col for col in df.columns if col not in exclude_cols]
    if extra_cols:
        agg_dict = {col: 'sum' if pd.api.types.is_numeric_dtype(df[col]) else 'first' for col in extra_cols}
        extra_features = df.groupby('from')[extra_cols].agg(agg_dict).add_prefix('from_')
        features_df = features_df.join(extra_features, how='left')
        numeric_cols = extra_features.select_dtypes(include='number').columns
        features_df[numeric_cols] = features_df[numeric_cols].fillna(0)

    int_cols = ['out_degree', 'in_degree', 'total_tx', 'net_flow']
    features_df[int_cols] = features_df[int_cols].astype(int)
    return features_df

# ========================================
# 6. 统一数据构建（保留所有原始信息）
# ========================================
ponzi_contract_data = {}
unique_addresses = ponzi_contract['address'].unique()

print("构建完整三模态 + 边列表 + 超图...")
for addr in tqdm(unique_addresses, desc="Contracts"):
    # 1. Trace Graph + Edge List
    trace_df, edge_list_str = construct_trace_graph_with_edges(ponzi_contract, addr)
    trace_features = extract_transactional_features(trace_df)

    # 2. Hypergraph
    hyper_df, hyperedge_dict = construct_hypergraph(ponzi_contract, addr, opcode_categories)
    hyper_features = extract_transactional_features(hyper_df)

    # 3. Opcode Sequence
    opcode_str = ponzi_contract.loc[ponzi_contract['address'] == addr, 'opcode'].iloc[0]
    opcode_list = opcode_str.split('\n')[0].split()
    chunks = chunk_opcodes_list_classified(opcode_list)
    stats = analyze_chunk_statistics(chunks)
    seq = generate_opcode_mapping_sequence(chunks)
    stats_text = "; ".join(
        f"{s['type']}: count={s['count']}, avg_len={s['avg_length']}, max={s['max_length']}, min={s['min_length']}"
        for s in stats
    )

    label = int(ponzi_contract.loc[ponzi_contract['address'] == addr, 'label'].iloc[0])

    ponzi_contract_data[addr] = {
        'trace_df': trace_df,
        'trace_features': trace_features,
        'edge_list_str': edge_list_str,
        'hyper_df': hyper_df,
        'hyper_features': hyper_features,
        'hyperedge_dict': hyperedge_dict,
        'opcode_stats': stats_text,
        'opcode_seq': seq,
        'original_opcode': opcode_str,
        'label': label
    }

# 特征对齐 + 合并
# 特征对齐 + 合并
def align_and_merge_features(data):
    all_cols = set()
    for d in data.values():
        all_cols.update(d['trace_features'].columns)
        all_cols.update(d['hyper_features'].columns)
    all_cols = sorted(all_cols)
    for addr, d in data.items():
        t = d['trace_features'].reindex(columns=all_cols, fill_value=0)
        h = d['hyper_features'].reindex(columns=all_cols, fill_value=0)
        merged = (t + h).astype('float32')
        d['node_features'] = merged
    return data

ponzi_contract_data = align_and_merge_features(ponzi_contract_data)

# ========================================
# 7. 统一 PyG Data（含所有信息）
# ========================================
def build_unified_data_list(data_dict):
    data_list = []
    for addr, d in data_dict.items():
        # 普通图
        if d['trace_df'].empty:
            G = nx.DiGraph(); G.add_node(0)
            node_count, edge_count, avg_degree = 1, 0, 0.0
        else:
            G = nx.from_pandas_edgelist(d['trace_df'], source='from', target='to', create_using=nx.DiGraph)
            pyg = from_networkx(G)
            edge_index = pyg.edge_index
            
            # 【✨ 新增：计算图拓扑特征来替换 Edge List】
            node_count = G.number_of_nodes()
            edge_count = G.number_of_edges()
            avg_degree = round(2 * edge_count / node_count, 2) if node_count > 0 else 0.0
            
        x = torch.tensor(d['node_features'].values, dtype=torch.float)

        # 超边
        hyperedges = []
        for edge_list in d['hyperedge_dict'].values():
            for nodes in edge_list:
                if len(nodes) >= 2:
                    hyperedges.append(nodes)
        if hyperedges:
            row = [i for e in hyperedges for i in e]
            col = [i for i, e in enumerate(hyperedges) for _ in e]
            hyperedge_index = torch.tensor([row, col], dtype=torch.long)
        else:
            hyperedge_index = torch.empty((2, 0), dtype=torch.long)

        y = torch.tensor([d['label']], dtype=torch.long)

        data = Data(
            x=x, edge_index=edge_index, hyperedge_index=hyperedge_index, y=y,
            addr=addr,
            # 【修改：不再保存 edge_list_str，而是保存图拓扑特征】
            graph_topo_features=f"Nodes={node_count}, Edges={edge_count}, AvgDegree={avg_degree}",
            opcode_stats=d['opcode_stats'],
            opcode_seq=d['opcode_seq'],
            original_opcode=d['original_opcode']
        )
        data.num_nodes = x.shape[0]
        data_list.append(data)
    return data_list

unified_data_list = build_unified_data_list(ponzi_contract_data)
temp_data, test_data = train_test_split(
    unified_data_list, test_size=0.3, random_state=42,
    stratify=[d.y.item() for d in unified_data_list]
)

# ========================================
# 8. 模型定义
# ========================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class GCNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.5):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        self.bns = torch.nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(GCNConv(hidden_dim, output_dim))
        self.bns.append(nn.BatchNorm1d(output_dim))
        self.classifier = nn.Sequential(
            nn.Linear(output_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch):
        h = x
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, edge_index)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        graph_emb = global_mean_pool(h, batch)
        logits = self.classifier(graph_emb)
        return logits, graph_emb

class HypergraphConv(nn.Module):
    def __init__(self, in_features, out_features, dropout=0.5):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X, H, D_v_inv, D_e_inv):
        HX = torch.sparse.mm(H.t(), X)
        D_e_inv_HX = torch.mm(D_e_inv, HX)
        Ht_D_e_inv_HX = torch.sparse.mm(H, D_e_inv_HX)
        X_out = torch.mm(D_v_inv, Ht_D_e_inv_HX)
        X_out = self.linear(X_out)
        X_out = F.relu(X_out)
        return self.dropout(X_out)

class HGCNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        self.convs.append(HypergraphConv(input_dim, hidden_dim, dropout))
        self.bns.append(nn.BatchNorm1d(hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(HypergraphConv(hidden_dim, hidden_dim, dropout))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        self.convs.append(HypergraphConv(hidden_dim, output_dim, dropout))
        self.bns.append(nn.BatchNorm1d(output_dim))
        self.classifier = nn.Sequential(
            nn.Linear(output_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 2)
        )

    def forward(self, X, H, D_v_inv, D_e_inv, batch=None):
        h = X
        for conv, bn in zip(self.convs, self.bns):
            h = conv(h, H, D_v_inv, D_e_inv)
            h = bn(h)
            h = F.relu(h)
            h = F.dropout(h, p=0.5, training=self.training)
        if batch is None:
            graph_emb = torch.mean(h, dim=0, keepdim=True)
        else:
            graph_emb = global_mean_pool(h, batch)
        logits = self.classifier(graph_emb)
        return logits, graph_emb

# ========================================
# 9. 加载模型 + 提取双嵌入
# ========================================
input_dim = unified_data_list[0].x.shape[1]
gcn_model = GCNEncoder(input_dim, 128, 8, 3, 0.5).to(DEVICE)
hgcn_model = HGCNN(input_dim, 256, 8, 3, 0.5).to(DEVICE)

gcn_model.load_state_dict(torch.load('best_ep_G_CE_loss.pth', map_location=DEVICE))
hgcn_model.load_state_dict(torch.load('best_hgcn.pt', map_location=DEVICE))

@torch.no_grad()
def extract_dual_embeddings(data_list, gcn, hgcn, device):
    gcn.eval(); hgcn.eval()
    gcn_embs, hgcn_embs = [], []
    for data in data_list:
        data = data.to(device)
        # GCN
        batch = torch.zeros(data.num_nodes, dtype=torch.long, device=device)
        _, gcn_vec = gcn(data.x, data.edge_index, batch)

        # HGCNN
        if data.hyperedge_index.size(1) == 0:
            hgcn_vec = torch.mean(data.x, dim=0, keepdim=True)
        else:
            row, col = data.hyperedge_index
            val = torch.ones(row.size(0), device=device)
            H = torch.sparse_coo_tensor(torch.stack([row, col]), val,
                                        size=(data.num_nodes, data.hyperedge_index[1].max().item()+1))
            Dv = torch.zeros(data.num_nodes, device=device)
            Dv.scatter_add_(0, row, torch.ones_like(row, dtype=torch.float))
            Dv_inv = torch.diag(1.0 / (Dv + 1e-8))
            De = torch.bincount(col, minlength=H.size(1)).to(torch.float)
            De_inv = torch.diag(1.0 / (De + 1e-8))
            _, hgcn_vec = hgcn(data.x, H, Dv_inv, De_inv, batch)

        gcn_embs.append(gcn_vec.cpu())
        hgcn_embs.append(hgcn_vec.cpu())
    return torch.cat(gcn_embs), torch.cat(hgcn_embs)

print("提取双嵌入...")
gcn_emb_all, hgcn_emb_all = extract_dual_embeddings(unified_data_list, gcn_model, hgcn_model, DEVICE)

# ========================================
# 9. 完整 Prompt 构建（保留原始所有字段 + HGCNN）
# ========================================
UNIFIED_INSTRUCTION = (
    "Based on the provided Ethereum smart contract's features, "
    "determine if this contract is a Ponzi Scheme. Respond only with 'True' or 'False'. "
    "The features include:\n"
    "1. **GCN Graph Embedding** – control flow trace summary.\n"
    "2. **Trace Flow Graph Topology** – structural complexity (Nodes, Edges, Avg. Degree).\n"
    "3. **HGCNN Hypergraph Embedding** – opcode category hyperedges.\n"
    "4. **Opcode Block Statistics** – count and length of semantic blocks.\n"
    "5. **Symbolic Opcode Sequence** – high-level block symbols."
)

def build_complete_prompt(data_subset, gcn_emb, hgcn_emb, data_list):
    prompts = []
    # 由于 train_test_split 已经完成，我们需要重新计算索引
    # 找到 data_subset 中每个元素在 data_list 中的原始索引
    data_list_map = {id(data): i for i, data in enumerate(data_list)}
    
    for i, data in enumerate(data_subset):
        # 使用 id() 确保获取正确的原始索引
        idx = data_list_map[id(data)]
        gcn_str = " ".join(f"{v:.6f}" for v in gcn_emb[idx].tolist())
        hgcn_str = " ".join(f"{v:.6f}" for v in hgcn_emb[idx].tolist())

        input_text = (
            "---CONTRACT FEATURES---\n"
            f"1. **GCN Graph Embedding (GNN Summary Vector):** {gcn_str}\n"
            # 【修改：用图拓扑特征替换 Edge List】
            f"2. **Trace Flow Graph Topology (Struct):** {data.graph_topo_features}\n"
            f"3. **HGCNN Hypergraph Embedding:** {hgcn_str}\n"
            # f"4. **Opcode Block Statistics:** {data.opcode_stats}\n"
            # f"5. **Symbolic Opcode Sequence:** {data.opcode_seq}\n"
            f"6. **Original Opcode Sequence:** {data.original_opcode}\n"
            "---END OF FEATURES---"
        )

        prompts.append({
            "instruction": UNIFIED_INSTRUCTION.strip(),
            "input": input_text.strip(),
            "output": "True" if data.y.item() == 1 else "False"
        })
    return prompts

train_prompt = build_complete_prompt(temp_data, gcn_emb_all, hgcn_emb_all, unified_data_list)
test_prompt  = build_complete_prompt(test_data , gcn_emb_all, hgcn_emb_all, unified_data_list)

with open("train_complete(original)_prompt.json", "w", encoding="utf-8") as f:
    json.dump(train_prompt, f, ensure_ascii=False, indent=2)
with open("test_complete_prompt(original).json",  "w", encoding="utf-8") as f:
    json.dump(test_prompt,  f, ensure_ascii=False, indent=2)

print(f"完成！训练集: {len(train_prompt)}, 测试集: {len(test_prompt)}")

d:\Tool\Anaconda\envs\ponzi_e\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


清理后样本数: 6468
构建完整三模态 + 边列表 + 超图...


Contracts: 100%|██████████| 6468/6468 [08:34<00:00, 12.57it/s]


提取双嵌入...
完成！训练集: 4527, 测试集: 1941


In [2]:
test_prompt[0]

{'instruction': "Based on the provided Ethereum smart contract's features, determine if this contract is a Ponzi Scheme. Respond only with 'True' or 'False'. The features include:\n1. **GCN Graph Embedding** – control flow trace summary.\n2. **Trace Flow Graph Topology** – structural complexity (Nodes, Edges, Avg. Degree).\n3. **HGCNN Hypergraph Embedding** – opcode category hyperedges.\n4. **Opcode Block Statistics** – count and length of semantic blocks.\n5. **Symbolic Opcode Sequence** – high-level block symbols.",
 'input': '---CONTRACT FEATURES---\n1. **GCN Graph Embedding (GNN Summary Vector):** 0.035504 1.074700 0.298881 0.912808 1.091569 1.005984 1.124897 0.094114\n2. **Trace Flow Graph Topology (Struct):** Nodes=42, Edges=157, AvgDegree=7.48\n3. **HGCNN Hypergraph Embedding:** 0.264583 0.310785 0.378408 0.123061 0.140309 0.000000 0.000000 0.328596\n6. **Original Opcode Sequence:** PUSH1 PUSH1 MSTORE PUSH4 PUSH1 PUSH1 EXP PUSH1 CALLDATALOAD DIV AND PUSH4 DUP2 EQ PUSH2 JUMPI JUM